In [5]:
import os

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score

path = os.getenv("DATA_PATH")
df = pd.read_csv(path)
target = "Problem_SKU"
seed = 1337 
# One-hot encode Storage_Size (drop size_4 as baseline)
size_dummies = pd.get_dummies(df['Storage_Size'], prefix='Size', drop_first=True).astype(int)

# Encode Defect_In_Linked_Receive as 0/1
defect_linked_num = df['Defect_In_Linked_Receive'].astype(int)

# Numeric features (keep standardized)
numeric_features = [
    "Global_SKU_Defect_Rate_%_std",
    "ABS_Volume_Difference_std",
    "Aisle_Hold_%_std",
    "#_Pick_Events_std",
    "#_Pick_Events_In_Clique_std",
    "#_Picks_std",
    "#_Picks_In_Clique_std",
    "Time_In_Loc_std",
    "Current_Max_Volume_std",
]

feature_cols = numeric_features + list(size_dummies.columns) + ['Defect_In_Linked_Receive']

# Combine all properly encoded features
X = df[numeric_features].copy()
X = pd.concat([X, size_dummies, defect_linked_num], axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

xgb = XGBClassifier(
    objective="binary:logistic",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=seed,
    eval_metric="logloss",
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

fi_xgb = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_xgb)

results = pd.DataFrame({
    "model": [#"DecisionTree", 
              "XGBoost"],
    "roc_auc": [#roc_auc_score(y_test, y_proba_xgb),
                roc_auc_score(y_test, y_proba_xgb)],
})
print(results)

XGBoost
              precision    recall  f1-score   support

       False      0.946     0.990     0.968     50954
        True      0.703     0.283     0.404      4046

    accuracy                          0.938     55000
   macro avg      0.824     0.637     0.686     55000
weighted avg      0.928     0.938     0.926     55000

ROC AUC: 0.8742971571520675
#_Picks_In_Clique_std           0.291325
Aisle_Hold_%_std                0.122333
Current_Max_Volume_std          0.108169
#_Picks_std                     0.079347
Defect_In_Linked_Receive        0.061832
ABS_Volume_Difference_std       0.059224
#_Pick_Events_In_Clique_std     0.055219
Size_size_2                     0.054648
Size_size_3                     0.051175
Size_size_4                     0.048084
Global_SKU_Defect_Rate_%_std    0.043279
#_Pick_Events_std               0.014170
Time_In_Loc_std                 0.011194
dtype: float32
     model   roc_auc
0  XGBoost  0.874297


In [3]:
from xgboost import XGBClassifier

xgb_recall = XGBClassifier(
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),  # ← imbalance ratio
    max_depth=6,
    n_estimators=300
)

xgb_recall.fit(X_train, y_train)

y_pred_xgb = xgb_recall.predict(X_test)
y_proba_xgb = xgb_recall.predict_proba(X_test)[:, 1]

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

fi_xgb = pd.Series(xgb_recall.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_xgb)



XGBoost
              precision    recall  f1-score   support

       False      0.968     0.889     0.927     50954
        True      0.312     0.633     0.418      4046

    accuracy                          0.870     55000
   macro avg      0.640     0.761     0.672     55000
weighted avg      0.920     0.870     0.890     55000

ROC AUC: 0.8458725219306003
#_Picks_In_Clique_std           0.171069
Defect_In_Linked_Receive        0.129783
Size_size_4                     0.090412
Size_size_2                     0.085010
Aisle_Hold_%_std                0.084604
Size_size_3                     0.080949
Current_Max_Volume_std          0.067610
#_Picks_std                     0.064731
ABS_Volume_Difference_std       0.054344
#_Pick_Events_In_Clique_std     0.054027
Global_SKU_Defect_Rate_%_std    0.048493
#_Pick_Events_std               0.034562
Time_In_Loc_std                 0.034407
dtype: float32
